# Stage 1 model

In [1]:
import pandas as pd
import plotly.express as px

from ambdes import Model, Runner, SimConfig, ambsys

## Run model for 100 minutes with log messages

TODO: Convert this into logging tests which compare patient results with record in the vidigi logger.

In [2]:
ambsys_data = ambsys(
    csv_path="data/AmbSYS-to-Mar-2026-UI7FG.csv",
    org_code="RYF",
    month="3",
    year=2026,
)
ambsys_data

{'mean_iat_min': {'C1': 5.02646098412341,
  'C2': 1.042868823735545,
  'C3': 2.4551754482455177,
  'C4': 135.6838905775076},
 'mean_handover_time_min': 29.916666666666668,
 'p90_handover_time_min': 51.28333333333333,
 'sd_handover_time_min': 11.385314934097805}

In [3]:
config = SimConfig(
    ambsys_data=ambsys_data,
)
config.n_ambulances = 1
model = Model(run_number=0, config=config)
model.run()

In [4]:
log = model.logger.to_dataframe()

In [5]:
print(model.patients[0].__dict__)
log[log["entity_id"] == 1]

{'patient_id': 1, 'category': 'C3', 'call_timestamp': 100.62352324131702, 'response_time': None}


,entity_id,event_type,event,time,run_number,resource_id
0,1,arrival_departure,arrival,0.837533,0,NaN
1,1,queue,ambulance_wait_begins,0.837533,0,NaN
2,1,resource_use,ambulance_arrives,0.837533,0,1.0
271,1,resource_use_end,ambulance_available,96.261361,0,1.0
272,1,arrival_departure,depart,96.261361,0,NaN
294,1,arrival_departure,arrival,100.623523,0,NaN
295,1,queue,ambulance_wait_begins,100.623523,0,NaN


In [6]:
print(model.patients[1].__dict__)
log[log["entity_id"] == 2]

{'patient_id': 2, 'category': 'C3', 'call_timestamp': 101.96697577951335, 'response_time': None}


,entity_id,event_type,event,time,run_number,resource_id
3,2,arrival_departure,arrival,1.235910,0,NaN
4,2,queue,ambulance_wait_begins,1.235910,0,NaN
273,2,resource_use,ambulance_arrives,96.261361,0,1.0
296,2,arrival_departure,arrival,101.966976,0,NaN
297,2,queue,ambulance_wait_begins,101.966976,0,NaN
616,2,resource_use_end,ambulance_available,198.807053,0,1.0
617,2,arrival_departure,depart,198.807053,0,NaN


## Run model for longer and inspect patient times

In [7]:
config = SimConfig(
    ambsys_data=ambsys_data,
    warm_up_period=0,
    data_collection_period=10080,  # One week
)
model = Model(run_number=0, config=config)
model.run()

In [8]:
df = pd.DataFrame(
    {
        "response_time": [p.response_time for p in model.patients],
        "category": [p.category for p in model.patients],
    }
)

fig = px.histogram(
    df,
    x="response_time",
    facet_col="category",
    nbins=50,
    category_orders={"category": ["C1", "C2", "C3", "C4"]},
    labels={
        "response_time": "Response time (minutes)",
        "category": "Category",
    },
    title="Distribution of response times by category",
)

# fig.update_yaxes(
#    matches=None,
#    showticklabels=True,
# )
fig.layout.yaxis.title.text = "Number of patients"

fig.show()

In [9]:
for cat in ["C1", "C2", "C3", "C4"]:
    fig = px.histogram(
        df[df["category"] == cat],
        x="response_time",
        nbins=20,
        title=f"Response times: {cat}",
        labels={"response_time": "Response time (minutes)"},
    )
    fig.update_yaxes(title_text="Number of patients")
    fig.show()

## Average results

In [10]:
runner = Runner(config)

In [11]:
results = runner.run_reps()

In [12]:
results["patients"]

,run,patient_id,category,call_timestamp,response_time
0,0,1,C2,0.837533,4.073027
1,0,2,C2,1.235910,7.424848
2,0,3,C2,1.615698,3.004763
3,0,4,C2,2.167870,1.554013
4,0,5,C3,3.757094,7.215603
...,...,...,...,...,...
79223,4,15974,C2,10076.322889,NaN
79224,4,15975,C2,10076.676822,1.634170
79225,4,15976,C2,10076.729948,NaN
79226,4,15977,C2,10077.762373,0.346001


In [13]:
results["run"]

,category,n_patients,mean_response_time,run
0,C1,2016,10.043064,0
1,C2,9861,9.874453,0
2,C3,3956,9.786215,0
3,C4,76,9.607152,0
4,C1,2024,10.011695,1
5,C2,9570,9.819834,1
6,C3,4150,10.001758,1
7,C4,69,9.693440,1
8,C1,2027,10.179896,2
9,C2,9662,10.040621,2


In [14]:
results["overall"]

,category,mean_n_patients,mean_response_time
0,C1,2013.4,9.957002
1,C2,9694.4,9.913094
2,C3,4057.8,10.029666
3,C4,80.0,10.412481


In [15]:
config.n_ambulances = 1
config.data_collection_period = 50_000
config.log_to_console = False
runner = Runner(config)
results = runner.run_single(run_number=0)

In [16]:
results["run"]

,category,n_patients,mean_response_time,run
0,C1,10122,24882.128638,0
1,C2,47831,24981.944913,0
2,C3,20204,25145.029568,0
3,C4,392,33869.408274,0


In [17]:
results["patients"].head(20)

,run,patient_id,category,call_timestamp,response_time
0,0,1,C2,0.837533,4.073027
1,0,2,C2,1.235910,102.450299
2,0,3,C2,1.615698,200.196118
3,0,4,C2,2.167870,306.023344
4,0,5,C3,3.757094,405.667583
5,0,6,C3,4.039516,522.338683
6,0,7,C2,5.075784,619.716820
7,0,8,C3,6.931662,745.773423
8,0,9,C2,7.120883,847.157297
9,0,10,C2,8.110538,949.657591


In [18]:
results["model"].logger.to_dataframe().head(30)

,entity_id,event_type,event,time,run_number,resource_id
0,1,arrival_departure,arrival,0.837533,0,NaN
1,1,queue,ambulance_wait_begins,0.837533,0,NaN
2,1,resource_use,ambulance_arrives,0.837533,0,1.0
3,2,arrival_departure,arrival,1.235910,0,NaN
4,2,queue,ambulance_wait_begins,1.235910,0,NaN
5,3,arrival_departure,arrival,1.615698,0,NaN
6,3,queue,ambulance_wait_begins,1.615698,0,NaN
7,4,arrival_departure,arrival,2.167870,0,NaN
8,4,queue,ambulance_wait_begins,2.167870,0,NaN
9,5,arrival_departure,arrival,3.757094,0,NaN
